# Mock Exam 1 — Practical Solutions

## Exercise 1 — Static scraping



In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"
CATALOGUE_URL = urljoin(BASE_URL, "catalogue/page-1.html")

headers = {
    "User-Agent": "Mozilla/5.0"}

resp = requests.get(CATALOGUE_URL, headers=headers, timeout=30)
resp.raise_for_status()
html = resp.text

soup = BeautifulSoup(html, "html.parser")
len(html), resp.status_code

In [ ]:
books = []

for pod in soup.select("article.product_pod"):
    a = pod.select_one("h3 a")
    rel_detail = a.get("href")
    detail_url = urljoin(CATALOGUE_URL, rel_detail)

    title = a.get("title", "").strip()
    price = pod.select_one(".price_color").get_text(strip=True)
    availability = pod.select_one(".availability").get_text(" ", strip=True)

    # Fetch product page to get category from breadcrumb
    d_resp = requests.get(detail_url, headers=headers, timeout=30)
    d_resp.raise_for_status()
    d_soup = BeautifulSoup(d_resp.text, "html.parser")

    # breadcrumb: Home > Books > Category > Title
    bc = d_soup.select("ul.breadcrumb li a")
    category = None
    if bc and len(bc) >= 3:
        category = bc[2].get_text(strip=True)

    books.append({
        "title": title,
        "price_raw": price,
        "availability_raw": availability,
        "category_name": category,
        "detail_url": detail_url
    })

df = pd.DataFrame(books)
df.head()

## Exercise 2 — Regex cleaning
Remove currency symbol and convert to numeric values.


In [ ]:
df['price'] = df['price'].str.replace('£', '', regex=True).astype(float)
df['price_eur'] = df['price'] * 1.15
df

## Exercise 3 — Filtering with str.contains
Keep only books that are in stock.


In [ ]:
df_in_stock = df[df['availability'].str.contains('In stock')]
df_in_stock[['title', 'price']]

## Exercise 4 — Pandas merge
Left join to keep all books even if category information is missing.


In [ ]:
df_categories = pd.DataFrame({
    'category_id': [1, 2],
    'category_name': ['Travel', 'Mystery']
})

df_books = df.copy()
df_books['category_id'] = [1, 2, 1]

merged = pd.merge(df_books, df_categories, how='left', on='category_id')
merged